# EEG Cohort Audit

**Term Project — EEG data availability and extraction QC**

This notebook audits the existing EEG band-power CSVs and participant processing summaries. It does **not** reprocess raw EEG or perform inferential statistics.

It checks participant/session/task coverage, expected recording conditions, channel and band completeness, missing power values, and duplicate extraction keys.

**Inputs:** the eight EEG results/processing-summary CSVs should be in the same folder as this notebook (or in `/mnt/data`).

**Outputs:** six audit CSV files are written to the working data folder.

In [1]:
from pathlib import Path
import pandas as pd

# Locate the input CSVs: prefer the notebook's working directory, otherwise /mnt/data.
DATA_DIR = Path.cwd()
required_probe = 'eeg_analysis_results_math_control.csv'
if not (DATA_DIR / required_probe).exists() and (Path('/mnt/data') / required_probe).exists():
    DATA_DIR = Path('/mnt/data')

print(f"Data folder: {DATA_DIR.resolve()}")

Data folder: /Users/ananyosen/Documents/Term Project/analysis


In [2]:
# Input files and expected structure
FILES = {
    "Math Control": ("eeg_analysis_results_math_control.csv", "subject_processing_summary_math_control.csv"),
    "Math Experiment": ("eeg_analysis_results_math_experiment.csv", "subject_processing_summary_math_experiment.csv"),
    "Reading Control": ("eeg_analysis_results_read_control.csv", "subject_processing_summary_read_control.csv"),
    "Reading Experiment": ("eeg_analysis_results_read_experiment.csv", "subject_processing_summary_read_experiment.csv"),
}

EXPECTED_CHANNELS = 32
EXPECTED_BANDS = 5
EXPECTED_POWER_ROWS_PER_RECORDING = EXPECTED_CHANNELS * EXPECTED_BANDS
EXPECTED_SESSIONS = ["ses-1", "ses-2"]

# Verify that every required input file is present before proceeding.
missing_inputs = [fname for pair in FILES.values() for fname in pair if not (DATA_DIR / fname).exists()]
if missing_inputs:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(missing_inputs))
print("All eight input files found.")

FileNotFoundError: Missing input files:
eeg_analysis_results_math_control.csv
subject_processing_summary_math_control.csv
eeg_analysis_results_math_experiment.csv
subject_processing_summary_math_experiment.csv
eeg_analysis_results_read_control.csv
subject_processing_summary_read_control.csv
eeg_analysis_results_read_experiment.csv
subject_processing_summary_read_experiment.csv

In [ ]:
# Load the four result files and four processing summaries
loaded = {}
for task_group, (result_name, summary_name) in FILES.items():
    results = pd.read_csv(DATA_DIR / result_name)
    summary = pd.read_csv(DATA_DIR / summary_name)
    loaded[task_group] = {"results": results, "summary": summary}
    print(f"{task_group}: {len(results):,} power rows; {summary['subject_id'].nunique()} participant IDs in processing summary")

In [ ]:
# Run the cohort and extraction-quality audit
subject_rows = []
taskgroup_rows = []
missing_rows = []
duplicate_rows = []
condition_rows = []
summary_status_rows = []

for task_group, bundle in loaded.items():
    df = bundle["results"]
    summ = bundle["summary"]
    task, group = task_group.split(" ", 1)
    extracted_ids = set(df["subject_id"].dropna().unique())
    summary_extracted_ids = set(summ.loc[summ["status"].eq("EXTRACTED"), "subject_id"])
    all_ids = sorted(set(summ["subject_id"].dropna()) | extracted_ids)

    expected_conditions = ["Baseline", "Activity1", "Activity2"] if task == "Math" else ["Baseline", "Activity"]
    expected_key = ["subject_id", "session", "run", "condition", "channel", "band"]
    duplicate_count = int(df.duplicated(expected_key, keep=False).sum())
    duplicate_rows.append({
        "task": task, "group": group, "result_file": FILES[task_group][0],
        "duplicate_key_rows": duplicate_count, "duplicate_key_definition": "|".join(expected_key)
    })

    for _, srow in summ.iterrows():
        summary_status_rows.append({
            "task": task, "group": group, "subject_id": srow["subject_id"],
            "processing_status": srow["status"],
            "recordings_loaded": srow.get("recordings_loaded"),
            "band_power_rows_summary": srow.get("band_power_rows"),
            "events_files_complete": all(bool(srow.get(col, False)) for col in summ.columns if col.endswith("_events_file")),
            "reason": srow.get("missing_or_inconsistent_reason")
        })

    for sid in all_ids:
        sub = df[df["subject_id"].eq(sid)]
        summary_row = summ[summ["subject_id"].eq(sid)]
        status = summary_row["status"].iloc[0] if len(summary_row) else "NOT_IN_SUMMARY"
        present_sessions = sorted(sub["session"].dropna().unique().tolist())
        present_conditions = sorted(sub["condition"].dropna().unique().tolist())
        recordings = sub[["session", "run", "condition"]].drop_duplicates()
        missing_rec = []
        for ses in EXPECTED_SESSIONS:
            for cond in expected_conditions:
                if not ((sub["session"] == ses) & (sub["condition"] == cond)).any():
                    missing_rec.append(f"{ses}:{cond}")
                    missing_rows.append({"task": task, "group": group, "subject_id": sid, "missing_session_condition": f"{ses}:{cond}"})

        per_record = sub.groupby(["session", "run", "condition"], dropna=False).agg(
            rows=("band", "size"), channels=("channel", "nunique"), bands=("band", "nunique"),
            null_abs=("absolute_power", lambda x: int(x.isna().sum())),
            null_rel=("relative_power", lambda x: int(x.isna().sum())),
        ).reset_index()
        record_complete = bool(
            len(per_record) > 0
            and (per_record["rows"] == EXPECTED_POWER_ROWS_PER_RECORDING).all()
            and (per_record["channels"] == EXPECTED_CHANNELS).all()
            and (per_record["bands"] == EXPECTED_BANDS).all()
            and (per_record["null_abs"] == 0).all()
            and (per_record["null_rel"] == 0).all()
        )

        for _, recrow in per_record.iterrows():
            condition_rows.append({
                "task": task, "group": group, "subject_id": sid,
                "session": recrow["session"], "run": recrow["run"], "condition": recrow["condition"],
                "rows": recrow["rows"], "unique_channels": recrow["channels"], "unique_bands": recrow["bands"],
                "missing_absolute_power": recrow["null_abs"], "missing_relative_power": recrow["null_rel"],
                "recording_complete_160_rows": recrow["rows"] == EXPECTED_POWER_ROWS_PER_RECORDING
                    and recrow["channels"] == EXPECTED_CHANNELS and recrow["bands"] == EXPECTED_BANDS
                    and recrow["null_abs"] == 0 and recrow["null_rel"] == 0
            })

        expected_recordings = len(EXPECTED_SESSIONS) * len(expected_conditions)
        both_sessions = set(EXPECTED_SESSIONS).issubset(set(present_sessions))
        subject_rows.append({
            "task": task, "group": group, "subject_id": sid, "processing_status": status,
            "in_result_file": sid in extracted_ids, "in_summary_extracted_status": sid in summary_extracted_ids,
            "sessions_present": ";".join(present_sessions), "both_PRE_POST_present": both_sessions,
            "conditions_present": ";".join(present_conditions), "recordings_present": len(recordings),
            "expected_recordings_for_full_task": expected_recordings, "full_session_condition_coverage": len(missing_rec) == 0,
            "missing_session_conditions": ";".join(missing_rec), "record_level_channel_band_power_complete": record_complete,
            "duplicate_rows_for_subject": int(sub.duplicated(expected_key, keep=False).sum()),
            "absolute_power_missing": int(sub["absolute_power"].isna().sum()),
            "relative_power_missing": int(sub["relative_power"].isna().sum()), "rows_in_result_file": len(sub),
        })

    taskgroup_rows.append({
        "task": task, "group": group, "participants_in_processing_summary": len(summ),
        "participants_EXTRACTED": int((summ["status"] == "EXTRACTED").sum()),
        "participants_SKIPPED": int((summ["status"] == "SKIPPED").sum()),
        "participants_in_results": len(extracted_ids), "result_rows": len(df),
        "sessions": ";".join(sorted(df["session"].unique())), "conditions": ";".join(sorted(df["condition"].unique())),
        "unique_channels": df["channel"].nunique(), "unique_bands": df["band"].nunique(),
        "channel_count_values": ";".join(map(str, sorted(df["channel_count"].dropna().unique()))),
        "missing_absolute_power": int(df["absolute_power"].isna().sum()),
        "missing_relative_power": int(df["relative_power"].isna().sum()), "duplicate_key_rows": duplicate_count,
        "summary_extracted_not_in_results": ";".join(sorted(summary_extracted_ids - extracted_ids)),
        "results_not_summary_extracted": ";".join(sorted(extracted_ids - summary_extracted_ids)),
    })

print("Audit computation finished.")

In [ ]:
# Save audit outputs
outputs = {
    "eeg_cohort_audit.csv": pd.DataFrame(subject_rows),
    "eeg_cohort_summary.csv": pd.DataFrame(taskgroup_rows),
    "eeg_missing_data_audit.csv": pd.DataFrame(missing_rows, columns=["task", "group", "subject_id", "missing_session_condition"]),
    "eeg_duplicate_key_audit.csv": pd.DataFrame(duplicate_rows),
    "eeg_condition_audit.csv": pd.DataFrame(condition_rows),
    "eeg_processing_status_audit.csv": pd.DataFrame(summary_status_rows),
}
for filename, table in outputs.items():
    table.to_csv(DATA_DIR / filename, index=False)
    print(f"Saved: {filename} ({len(table):,} rows)")

In [ ]:
# Display key cohort summary and QC indicators
cohort_summary = outputs["eeg_cohort_summary.csv"]
display(cohort_summary)

print("\nDuplicate key audit:")
display(outputs["eeg_duplicate_key_audit.csv"])

print("\nMissing-data audit (first 25 rows):")
display(outputs["eeg_missing_data_audit.csv"].head(25))

## Interpretation notes

- This audit verifies the structure and availability of exported EEG band-power data; it is **not** a signal-quality assessment of raw EEG.
- A participant with missing task/session data should not be automatically excluded from every analysis. Cohort inclusion is outcome- and comparison-specific.
- This notebook does not perform hypothesis tests. The next project step is to define participant-level EEG aggregation and primary outcomes before running statistical analyses.
- Keep the input files, this notebook, and the resulting audit CSVs together for reproducibility.